# S4 · Ejercicio 4 — Estratificación socioeconómica con K-Means

**Curso de Big Data · Dr. Abel Coronado** · Semana 4

> ⚙️ **Esta variante es para: Container (Podman) / Vagrant — Spark 4.0 · Scala 2.13.**  
> Comprueba tu versión con `import pyspark; pyspark.__version__`. Si NO coincide, usa la carpeta de tu plataforma (`container/` o `portable/`).

🎯 **Objetivo.** Agrupar manzanas del Censo en K=5 estratos con K-Means (no supervisado) y exportar el resultado.

📦 **Datos.** `/data/raw/geodatos_mexico/censo_2020_nacional.parquet`.

✅ **Prerrequisitos.** Censo en HDFS. Requiere geopandas (pip install geopandas).

📝 **Entregable.** Estratificación en HDFS y/o GeoPackage exportado.

⬇️ **Descarga (Hugging Face):** https://huggingface.co/datasets/abxda/bdp-lab/resolve/main/cuadernos/semana_4/container/01_ClasificacionNoSupervisada_KMeans.ipynb

💬 **¿Un error?** Captura (celda + mensaje) y repórtalo por **Blackboard** (indica SO y paso).

---

> Lee los comentarios de cada celda: explican el porqué de cada paso.


In [ ]:
# =============================================================================
# Celda 1: Inicialización de Spark y Sedona
# =============================================================================
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when
from sedona.spark import SedonaContext

# Detener cualquier sesión previa para una inicialización limpia
try:
    spark.stop()
    print("Sesión de Spark anterior detenida.")
except:
    pass

print("» Paso 1: Configurando la sesión de Spark para clustering geoespacial...")
spark = (
    SparkSession.builder.appName("KMeans_Socioeconomico_HDFS")
    .config("spark.driver.memory", "2g")  # BDP v7: recorte de un estado
    .config("spark.locality.wait", "0")
    .config("spark.sql.shuffle.partitions", "200")
    # BDP v7: Sedona 1.9.1 ya viene pre-cargado en el laboratorio (sin descargas de Maven)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator")
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions")
    .config("spark.sql.ansi.enabled", "false").getOrCreate()
)

sedona = SedonaContext.create(spark)
print("✅ Sesión de Spark con Apache Sedona iniciada correctamente.")

In [ ]:
# =============================================================================
# Celda 2: Carga y Selección de Datos BASE (MODIFICADO para más robustez)
# =============================================================================
print("\n» Paso 2: Cargando las columnas BASE para calcular los indicadores.")

# --- 2.1 Definir ruta y variables base según el diccionario ---
CENSO_PATH_HDFS = "hdfs:///data/raw/geodatos_mexico/censo_2020_nacional.parquet"
variables_base_censo = [
    'CVEGEO', 'geometry',
    'PEA', 'P_12YMAS', 'P18YM_PB', 'P_18YMAS',
    'VIVPARH_CV', 'VPH_STVP', 'VPH_SPMVPI', 'VPH_CVJ'
]

# --- 2.2 Cargar y seleccionar ---
print(f"   Cargando Censo desde: {CENSO_PATH_HDFS}")
censo_completo_sdf = spark.read.format("geoparquet").load(CENSO_PATH_HDFS)
censo_base_sdf = censo_completo_sdf.select(variables_base_censo)

# --- 2.3 ¡NUEVO! Limpieza de NULOS ---
# Obtenemos la lista de variables numéricas para rellenar
variables_numericas = [v for v in variables_base_censo if v not in ['CVEGEO', 'geometry']]

# Rellenamos los valores nulos con 0 solo en las columnas numéricas
censo_base_sin_nulos_sdf = censo_base_sdf.na.fill(0, subset=variables_numericas)
print("   Valores nulos reemplazados por 0.")


print(f"✅ Datos base del Censo cargados y pre-limpiados. Total de manzanas: {censo_base_sin_nulos_sdf.count():,}")
censo_base_sin_nulos_sdf.show(5)

In [ ]:
# =============================================================================
# Celda 3: Limpieza de Datos Negativos (MODIFICADO)
# =============================================================================
print("\n» Paso 3: Limpiando los datos. Reemplazando valores negativos por 0.")

# La lista de variables a limpiar sigue siendo la misma
variables_a_limpiar = [v for v in variables_base_censo if v not in ['CVEGEO', 'geometry']]

# Ahora partimos del DataFrame que ya no tiene nulos
censo_limpio_sdf = censo_base_sin_nulos_sdf

# Iteramos y aplicamos la limpieza de negativos
for variable in variables_a_limpiar:
    censo_limpio_sdf = censo_limpio_sdf.withColumn(
        variable,
        when(col(variable) < 0, 0.0).otherwise(col(variable))
    )

print("✅ Datos base completamente limpios (sin nulos ni negativos) y listos para el cálculo.")

In [ ]:
# =============================================================================
# Celda 4: Cálculo de Indicadores (SIN CAMBIOS, pero ahora más seguro)
# =============================================================================
print("\n» Paso 4: Calculando los indicadores porcentuales.")

# Usamos el DataFrame completamente limpio para los cálculos
indicadores_sdf = censo_limpio_sdf.withColumn(
    "ECO1_R",
    when(col("P_12YMAS") > 0, (col("PEA") / col("P_12YMAS")) * 100).otherwise(0.0)
).withColumn(
    "EDU46_R",
    when(col("P_18YMAS") > 0, (col("P18YM_PB") / col("P_18YMAS")) * 100).otherwise(0.0)
).withColumn(
    "VIV82_R",
    when(col("VIVPARH_CV") > 0, (col("VPH_STVP") / col("VIVPARH_CV")) * 100).otherwise(0.0)
).withColumn(
    "VIV83_R",
    when(col("VIVPARH_CV") > 0, (col("VPH_SPMVPI") / col("VIVPARH_CV")) * 100).otherwise(0.0)
).withColumn(
    "VIV84_R",
    when(col("VIVPARH_CV") > 0, (col("VPH_CVJ") / col("VIVPARH_CV")) * 100).otherwise(0.0)
)

print("   Verificación de los indicadores calculados:")
indicadores_sdf.select("CVEGEO", "ECO1_R", "EDU46_R", "VIV82_R", "VIV83_R", "VIV84_R").show(5)
print("✅ Indicadores calculados exitosamente.")

In [ ]:
# =============================================================================
# Celda 5: Preparación para K-Means (VectorAssembler) (MODIFICADO)
# =============================================================================
from pyspark.ml.feature import VectorAssembler

print("\n» Paso 5: Ensamblando los indicadores calculados en un vector de 'features'.")

# Las columnas de entrada para el clustering ahora son nuestros indicadores calculados
variables_a_clusterizar = ['ECO1_R', 'EDU46_R', 'VIV82_R', 'VIV83_R', 'VIV84_R']

assembler = VectorAssembler(
    inputCols=variables_a_clusterizar,
    outputCol="features",
    handleInvalid="skip"
)

# Transformamos el DataFrame con los indicadores para añadir la columna 'features'
censo_preparado_sdf = assembler.transform(indicadores_sdf)

print("   Muestra del DataFrame con la nueva columna 'features':")
censo_preparado_sdf.select("CVEGEO", "features").show(5, truncate=False)
print("✅ Datos vectorizados y listos para K-Means.")

In [ ]:
# =============================================================================
# Celda 6: Entrenamiento del Modelo K-Means (SIN CAMBIOS)
# =============================================================================
from pyspark.ml.clustering import KMeans

print("\n» Paso 6: Entrenando el modelo K-Means. (Esto puede tardar)")

# Definir y entrenar el modelo
kmeans = KMeans(featuresCol="features", k=5, seed=1)
kmeans_model = kmeans.fit(censo_preparado_sdf)
print("✅ Modelo K-Means entrenado.")

# Asignar cada manzana a un clúster
censo_estratificado_sdf = kmeans_model.transform(censo_preparado_sdf)

print("\n   Muestra de manzanas con su clúster ('prediction') asignado:")
censo_estratificado_sdf.select("CVEGEO", "prediction").show(10)

In [ ]:
# =============================================================================
# Celda 7: Análisis e Interpretación de los Clústeres (SIN CAMBIOS EN LÓGICA)
# =============================================================================
print("\n» Paso 7: Analizando los perfiles de cada clúster.")

# Agrupamos por clúster y calculamos promedios
perfiles_clusters = censo_estratificado_sdf.groupBy("prediction").agg(
    F.mean("ECO1_R").alias("prom_pob_activa"),
    F.mean("EDU46_R").alias("prom_educ_superior"),
    F.mean("VIV82_R").alias("prom_tv_paga"),
    F.mean("VIV83_R").alias("prom_streaming"),
    F.mean("VIV84_R").alias("prom_videojuegos"),
    F.count("*").alias("num_manzanas")
).orderBy("prediction")

print("--- Perfiles de los 5 Estratos Socioeconómicos Encontrados (basados en indicadores calculados) ---")
perfiles_clusters.show(truncate=False)
print("✅ Análisis de clústeres completado.")

In [ ]:
# =============================================================================
# Celda 8: Guardado del Resultado Final (MODIFICADO)
# =============================================================================
print("\n» Paso 8: Guardando el resultado final en HDFS.")

# Seleccionamos las columnas que queremos en nuestro archivo final
columnas_finales = [
    "CVEGEO",
    "prediction", # El ID del clúster/estrato
    "ECO1_R", "EDU46_R", "VIV82_R", "VIV83_R", "VIV84_R", # Los indicadores que calculamos
    "geometry"
]
resultado_final_sdf = censo_estratificado_sdf.select(columnas_finales) \
    .withColumnRenamed("prediction", "estrato_socioeconomico")

# Definir la ruta de salida
OUTPUT_PATH_HDFS = "hdfs:///data/processed/estratificacion_socioeconomica_calculada.geoparquet"
print(f"   Guardando en: {OUTPUT_PATH_HDFS}")

# Guardar como GeoParquet
resultado_final_sdf.write.format("geoparquet").mode("overwrite").save(OUTPUT_PATH_HDFS)

print(f"✅ ¡Proceso completado! El GeoParquet con la estratificación calculada está listo en HDFS.")


In [ ]:
# =============================================================================
# Celda 9: Instalación de Librerías (si es necesario)
# =============================================================================
#python -m pip install geopandas pygeos

import geopandas as gpd
print("✅ Librerías para exportación listas.")

In [ ]:
# =============================================================================
# Celda 10: Definición de Niveles y Colores para los Clústeres
# =============================================================================
from pyspark.sql.functions import col, lit, when

print("\n» Paso 9: Asignando niveles socioeconómicos (Alto, Medio, Bajo) a los clústeres.")

# --- 1. Calcular el Score Socioeconómico ---
# Partimos del DataFrame 'perfiles_clusters' que ya calculamos
perfiles_con_score_sdf = perfiles_clusters.withColumn(
    "score_socioeconomico",
    (col("prom_pob_activa") + col("prom_educ_superior") + col("prom_tv_paga") + col("prom_streaming") + col("prom_videojuegos")) / 5
).orderBy(col("score_socioeconomico").desc())

print("   Perfiles de clústeres ordenados por score socioeconómico:")
perfiles_con_score_sdf.show()

# --- 2. Asignar Niveles y Colores ---
# Recolectamos los resultados ordenados al driver para una manipulación más sencilla (es una tabla muy pequeña)
perfiles_pandas = perfiles_con_score_sdf.toPandas()

# Creamos la asignación de nivel y color
# Para K=5: 2 Altos, 1 Medio, 2 Bajos
niveles = ['Alto', 'Alto', 'Medio', 'Bajo', 'Bajo']
colores = ['Verde', 'Verde', 'Amarillo', 'Rojo', 'Rojo']
perfiles_pandas['nivel'] = niveles
perfiles_pandas['color'] = colores

# Convertimos de vuelta a un DataFrame de Spark para poder hacer el join
niveles_sdf = spark.createDataFrame(perfiles_pandas)

print("\n--- Tabla Final de Perfiles con Nivel y Color Asignado ---")
niveles_sdf.select("prediction", "score_socioeconomico", "nivel", "color", "num_manzanas").show()

In [ ]:
# =============================================================================
# Celda 11: Unir Colores al DataFrame Principal
# =============================================================================
print("\n» Paso 11: Uniendo la información de nivel y color al DataFrame de manzanas.")

# El DataFrame 'resultado_final_sdf' ya está en memoria
# Unimos por el ID del clúster
manzanas_con_color_sdf = resultado_final_sdf.join(
    niveles_sdf.select("prediction", "nivel", "color"),
    resultado_final_sdf.estrato_socioeconomico == niveles_sdf.prediction,
    "left"
).drop("prediction") # Eliminamos la columna duplicada del join

print("   Muestra de las manzanas con su nuevo nivel y color:")
manzanas_con_color_sdf.select("CVEGEO", "estrato_socioeconomico", "nivel", "color").show(10)

In [ ]:
niveles_sdf.show(5)

In [ ]:
manzanas_con_color_sdf.show(5)

In [ ]:
# =============================================================================
# Celda 11: GUARDAR EL RESULTADO FINAL EN HDFS
# =============================================================================
print("\n» Paso 11: Guardando el resultado final con colores como GeoParquet en HDFS.")

# Define la ruta de salida en HDFS.
OUTPUT_HDFS_PATH = "/data/processed/estratificacion_colores_geoparquet"

print(f"   Guardando en la ruta de HDFS: {OUTPUT_HDFS_PATH}")

# .coalesce(1) une el resultado en un solo archivo Parquet dentro de la carpeta en HDFS.
manzanas_con_color_sdf.coalesce(1).write.format("geoparquet").mode("overwrite").save(OUTPUT_HDFS_PATH)

print(f"✅ Resultado guardado exitosamente en HDFS.")

# --- Detenemos la sesión de Spark ---
spark.stop()
print("✅ Sesión de Spark detenida.")

In [ ]:
# =============================================================================
# Celda 12: DESCARGA DE HDFS Y CONVERSIÓN A GEOPACKAGE (SCRIPT LOCAL)
# =============================================================================
# Este script se ejecuta localmente, sin necesidad de Spark.
# Asegúrate de que las librerías 'hdfs' y 'geopandas' están instaladas.

import os
from hdfs import InsecureClient
import geopandas as gpd

print("\n» Paso 12: Descargando resultado desde HDFS y convirtiendo a GeoPackage.")

# --- 1. Definición de Rutas ---
# Ruta de origen en HDFS (la que acabamos de guardar)
HDFS_SOURCE_PATH = "/data/processed/estratificacion_colores_geoparquet"

# Ruta de destino local. Se creará una carpeta con este nombre en el directorio actual.
# os.getcwd() asegura que sea en 'C:\BDP\notebooks' si ejecutas el notebook desde ahí.
LOCAL_DEST_PATH = os.path.join(os.getcwd(), "estratificacion_colores_geoparquet")

GEOPACKAGE_FILENAME = "estratificacion_nacional_colores.gpkg"

print(f"Directorio de origen en HDFS: {HDFS_SOURCE_PATH}")
print(f"Directorio de destino local: {LOCAL_DEST_PATH}")
print(f"Archivo de salida final: {GEOPACKAGE_FILENAME}")


# --- 2. Descarga desde HDFS ---
try:
    print("\nConectando al cliente HDFS...")
    # Conexión al NameNode (puerto 9870 es el estándar en BDPv4)
    client = InsecureClient('http://localhost:9870')

    print("Iniciando la descarga...")
    # overwrite=True borrará el directorio local si ya existe.
    client.download(HDFS_SOURCE_PATH, LOCAL_DEST_PATH, overwrite=True)
    print("✅ ¡Descarga desde HDFS completada exitosamente!")

except Exception as e:
    print(f"❌ Ocurrió un error durante la descarga: {e}")
    # Detenemos la ejecución si la descarga falla
    raise SystemExit("No se puede continuar sin los datos.")


# --- 3. Conversión a GeoPackage ---
print("\nIniciando conversión a GeoPackage...")
try:
    # Verificamos si la descarga fue exitosa antes de continuar
    if os.path.exists(LOCAL_DEST_PATH):
        # Leer el GeoParquet descargado con GeoPandas
        print(f"   Leyendo GeoParquet desde la carpeta local: {LOCAL_DEST_PATH}")
        manzanas_gdf = gpd.read_parquet(LOCAL_DEST_PATH)
        print(f"   Datos cargados en GeoPandas. Total de manzanas: {len(manzanas_gdf):,}")

        # Guardar como GeoPackage
        print(f"   Guardando como GeoPackage en: {GEOPACKAGE_FILENAME}")
        manzanas_gdf.to_file(GEOPACKAGE_FILENAME, driver="GPKG")

        print(f"\n✅ ¡Éxito! El archivo '{GEOPACKAGE_FILENAME}' ha sido creado.")
    else:
        print("❌ ERROR: El directorio local no fue encontrado después de la descarga.")

except Exception as e:
    print(f"❌ Ocurrió un error durante la conversión a GeoPackage: {e}")